# Mamíferos de Colombia — Semana 14: Modelos de ensamble frente al modelo destilado

**Integrantes:** Jaime Andrés Ortega Cadena · David Alejandro Muñoz Carmona

## Escenarios que se comparan
| Escenario | Qué es |
|---|---|
| Modelos individuales | Cada EfficientNetV2-S del proyecto por separado (v2 y v4) |
| **Ensamble por promedio** | Promedio simple de las probabilidades de los dos modelos (0,5 / 0,5) |
| **Ensamble ponderado** | Promedio con pesos distintos, elegidos con **validación** (y volteo de la foto, TTA, si mejora en validación) |
| **Modelo híbrido** | Ensamble ponderado (imagen) **+ ubicación** de la foto: combina la red neuronal con la información de qué especies viven en cada departamento |
| **Modelo destilado** | MobileNetV3-Large que aprendió del ensamble (destilación), en TFLite float16 |
| **Destilado + ubicación** | Lo que corre en la app del celular |

Se mide en cada escenario **accuracy, F1 macro, top-3, tamaño y latencia** (CPU, una foto a la vez, como en un celular).
Pesos, TTA y parámetros de la ubicación se eligen **solo con validación**; la prueba se usa una sola vez al final.

## Entradas en Kaggle (panel derecho → *Input*)
1. Dataset `mamiferos-colombia-40`.
2. Los dos modelos entrenados: `modelo-mamiferos-v2` y la salida del cuaderno `notebook5bd9742cb0` (traen `profesor_mejor.keras` y `division_dataset.csv`).
3. El modelo de la app `mamiferos.tflite`: la salida del cuaderno `notebook901900824d`, o un dataset con ese archivo.

*Accelerator* → GPU T4 · *Run All* (unos 10 minutos).

---
# 0–3. Configuración, ingesta, auditoría, división y lectura (idénticas al proyecto)

In [ ]:
# 0.1 CONFIGURACIÓN (la misma del proyecto)
import os, sys, io, json, time, math, random, shutil, zipfile, hashlib, logging, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")

EN_KAGGLE = Path("/kaggle/input").exists()

# ---------- Datos ----------
FRAC_VAL, FRAC_TEST = 0.15, 0.15       # validación y prueba: SOLO fotos de Colombia, separadas por registro (gbifID)
MAX_TRAIN_POR_CLASE = 1200             # tope en entrenamiento para que la ardilla colirroja (2.452) no domine

# ---------- Modelo profesor (Transfer Learning) ----------
MODELO_BASE = "EfficientNetV2S"
IMG_SIZE = 260
BATCH = 32
DENSA = 512                            # neuronas de la capa densa antes de la salida
DROPOUT_1, DROPOUT_2 = 0.40, 0.30      # anti-overfitting
REGULARIZACION, FACTOR_REG = "l2", 1e-4   # "l1", "l2", "l1_l2" o None  (penaliza los pesos grandes)
LABEL_SMOOTHING = 0.10

# ---------- Entrenamiento ----------
EPOCAS_ETAPA1 = 10                     # etapa 1: base congelada, solo se entrena la cabeza
EPOCAS_TOTALES = 60                    # etapa 2: ajuste fino de la parte alta de la base (épocas 11 a 60 = 50 épocas)
LR_ETAPA1, LR_ETAPA2 = 1e-3, 1e-4      # optimizador Adam
FRACCION_AJUSTE_FINO = 0.60            # se descongela el 60 % final de las capas de la base
PACIENCIA = 12                         # EarlyStopping: para si val_accuracy no mejora en 12 épocas
EPOCA_MINIMA = 50                      # EarlyStopping no puede detener el entrenamiento antes de la época 50
META_ACCURACY = 0.94

MODO_PRUEBA = False                    # True = corrida corta de demostración (pocos lotes, 2 épocas) para verificar el pipeline
SEED = 42
random.seed(SEED); np.random.seed(SEED)

print("Kaggle:", EN_KAGGLE, "| modelo:", MODELO_BASE, f"{IMG_SIZE}px | épocas: {EPOCAS_ETAPA1} + {EPOCAS_TOTALES - EPOCAS_ETAPA1} = {EPOCAS_TOTALES}",
      "| MODO_PRUEBA" if MODO_PRUEBA else "")

In [ ]:
# 0.2 Entorno: GPU, TensorFlow/Keras, carpetas de salida y LOG
import tensorflow as tf
import keras
from keras import layers, regularizers
from IPython.display import display

BASE_SALIDA = Path("/kaggle/working") if EN_KAGGLE else Path.cwd()
SALIDA = BASE_SALIDA / "ensamble_vs_destilado"
EVI = SALIDA / "evidencia"
for p in [SALIDA, EVI, SALIDA / "logs"]:
    p.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("ENTRENAMIENTO"); log.setLevel(logging.INFO); log.handlers.clear(); log.propagate = False
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S")
_fh = logging.FileHandler(SALIDA / "logs" / f"entrenamiento_{datetime.now():%Y%m%d_%H%M%S}.log", encoding="utf-8")
_sh = logging.StreamHandler(sys.stdout)
for h in (_fh, _sh):
    h.setFormatter(_fmt); log.addHandler(h)

GPUS = tf.config.list_physical_devices("GPU")
if GPUS:
    keras.mixed_precision.set_global_policy("mixed_float16")     # ~2x más rápido en GPU T4
keras.utils.set_random_seed(SEED)
log.info("=" * 70)
log.info(f"SESIÓN DE ENTRENAMIENTO | TensorFlow {tf.__version__} | Keras {keras.__version__} | "
         f"GPU: {[g.name for g in GPUS] or 'NINGUNA'} | precisión: {keras.mixed_precision.global_policy().name}")
log.info(f"Salidas en: {SALIDA}")
if not GPUS:
    print("⚠️  No hay GPU. Panel derecho → Session options → Accelerator: GPU T4 x2 (requiere celular verificado).")

In [ ]:
# 1.1 Ingesta: ubicar el dataset subido a Kaggle (Kaggle descomprime el ZIP solo) y leer metadata.csv
RAIZ = Path("/kaggle/input") if EN_KAGGLE else Path.cwd()
encontrados = sorted(RAIZ.rglob("metadata.csv"))
if encontrados:
    DATOS = encontrados[0].parent
else:                                                  # por si el ZIP no se descomprimió
    zips = sorted(RAIZ.rglob("*.zip"))
    assert zips, "No encuentro el dataset. Panel derecho → Input → Add Input → Your Datasets → mamiferos-colombia-40"
    DATOS = Path("/tmp/datos")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATOS)
meta = pd.read_csv(DATOS / "metadata.csv")
carpeta_img = DATOS / "dataset" if (DATOS / "dataset").exists() else DATOS
meta["ruta"] = meta["archivo"].map(lambda a: str(carpeta_img / a))
log.info(f"Dataset en: {DATOS}")
log.info(f"INGESTA: {len(meta)} imágenes | {meta.clase.nunique()} clases | "
         f"Colombia {int((meta.origen == 'colombia').sum())} | Latinoamérica {int((meta.origen != 'colombia').sum())}")

In [ ]:
# 1.2 Auditoría automática de la ingesta (limpieza antes de entrenar)
def auditar(ruta):
    p = Path(ruta)
    if not p.exists():
        return "no_existe", ""
    data = p.read_bytes()
    try:
        with Image.open(io.BytesIO(data)) as im:
            im.verify()
        with Image.open(io.BytesIO(data)) as im:
            w, h = im.size
    except Exception:
        return "corrupta", ""
    if min(w, h) < 64:
        return "muy_pequena", ""
    return "", hashlib.sha256(data).hexdigest()

t0 = time.time()
res = [auditar(r) for r in meta["ruta"]]
meta["problema"] = [r[0] for r in res]; meta["sha256"] = [r[1] for r in res]
dup = meta["sha256"].ne("") & meta.duplicated("sha256", keep="first")
meta.loc[dup, "problema"] = "duplicado_exacto"
aud = meta["problema"].replace("", "OK").value_counts()
log.info(f"AUDITORÍA ({time.time() - t0:.0f} s): " + " | ".join(f"{k}={v}" for k, v in aud.items()))
display(aud.to_frame("imágenes"))
meta = meta[meta["problema"] == ""].reset_index(drop=True)

CLASES = sorted(meta["clase"].unique())
N_CLASES = len(CLASES)
IDX = {c: i for i, c in enumerate(CLASES)}
meta["y"] = meta["clase"].map(IDX)
NOMBRE = meta.drop_duplicates("clase").set_index("clase")["nombre_comun"].to_dict()
log.info(f"Imágenes válidas para entrenar: {len(meta)} | clases: {N_CLASES}")

In [ ]:
# 2.1 División estratificada por clase y agrupada por registro GBIF
rng = np.random.default_rng(SEED)
meta["split"] = "train"
for c, d in meta.groupby("clase"):
    reg_fuera = set(d.loc[d["origen"] != "colombia", "gbifID"])     # registros de otros países: solo entrenamiento
    col = d[~d["gbifID"].isin(reg_fuera)]
    por_reg = col.groupby("gbifID").size()
    ids = por_reg.index.to_numpy().copy(); rng.shuffle(ids)
    total, acum, asign = int(por_reg.sum()), 0, {}
    for g in ids:
        asign[g] = "test" if acum < total * FRAC_TEST else ("val" if acum < total * (FRAC_TEST + FRAC_VAL) else "train")
        acum += int(por_reg[g])
    meta.loc[col.index, "split"] = col["gbifID"].map(asign).values

# Tope por clase en entrenamiento (se priorizan las fotos de Colombia)
meta["_azar"] = rng.random(len(meta))
tr = meta[meta.split == "train"].sort_values(["clase", "origen", "_azar"],
                                             key=lambda s: (s != "colombia") if s.name == "origen" else s)
sobran = tr.groupby("clase").cumcount() >= MAX_TRAIN_POR_CLASE
meta.loc[tr.index[sobran.values], "split"] = "sin_usar"

# Verificación: ningún registro GBIF puede estar en dos grupos
g = {s: set(meta.loc[meta.split == s, "gbifID"]) for s in ["train", "val", "test"]}
assert not (g["train"] & g["val"]) and not (g["train"] & g["test"]) and not (g["val"] & g["test"]), "¡Fuga de datos!"
assert (meta.loc[meta.split.isin(["val", "test"]), "origen"] == "colombia").all()

tabla_split = pd.crosstab(meta["clase"], meta["split"]).reindex(columns=["train", "val", "test", "sin_usar"], fill_value=0)
display(tabla_split)
tabla_split.to_csv(EVI / "division_por_clase.csv", encoding="utf-8-sig")
meta.drop(columns=["_azar"]).to_csv(SALIDA / "division_dataset.csv", index=False, encoding="utf-8-sig")
cuenta = meta["split"].value_counts()
log.info("DIVISIÓN por gbifID (sin fuga de datos ✓): " + " | ".join(f"{k}={cuenta.get(k, 0)}" for k in ["train", "val", "test", "sin_usar"]))

In [ ]:
# 2.2 Pesos por clase (compensan el desbalance): peso = N / (K · n_clase), limitado entre 0.5 y 5
n_tr = meta[meta.split == "train"].groupby("y").size().reindex(range(N_CLASES), fill_value=1)
PESOS = np.clip(n_tr.sum() / (N_CLASES * n_tr), 0.5, 5.0)
meta["peso"] = meta["y"].map(PESOS)
display(pd.DataFrame({"clase": CLASES, "imagenes_train": n_tr.values, "peso": PESOS.round(2).values})
          .sort_values("peso", ascending=False).head(10))
log.info(f"Pesos por clase: mínimo {PESOS.min():.2f} ({CLASES[int(PESOS.idxmin())]}) | "
         f"máximo {PESOS.max():.2f} ({CLASES[int(PESOS.idxmax())]})")

In [ ]:
# 3.1 Lectura eficiente con tf.data (paralela + prefetch)
AUTOTUNE = tf.data.AUTOTUNE

def leer_imagen(ruta, size):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    return tf.image.resize(img, (size, size), antialias=True)          # float32 en 0..255

def hacer_ds(df, size=IMG_SIZE, entrenamiento=False):
    if entrenamiento:
        ds = tf.data.Dataset.from_tensor_slices((df["ruta"].values, df["y"].values.astype("int32"),
                                                 df["peso"].values.astype("float32")))
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(lambda r, y, w: (leer_imagen(r, size), tf.one_hot(y, N_CLASES), w), num_parallel_calls=AUTOTUNE)
    else:
        ds = tf.data.Dataset.from_tensor_slices((df["ruta"].values, df["y"].values.astype("int32")))
        ds = ds.map(lambda r, y: (leer_imagen(r, size), tf.one_hot(y, N_CLASES)), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

TRAIN = meta[meta.split == "train"].reset_index(drop=True)
VAL = meta[meta.split == "val"].reset_index(drop=True)
TEST = meta[meta.split == "test"].reset_index(drop=True)
ds_train, ds_val, ds_test = hacer_ds(TRAIN, entrenamiento=True), hacer_ds(VAL), hacer_ds(TEST)
if MODO_PRUEBA:
    ds_train, ds_val, ds_test = ds_train.take(15), ds_val.take(5), ds_test.take(5)
log.info(f"tf.data listo | train {len(TRAIN)} | val {len(VAL)} | test {len(TEST)} | lote {BATCH} | "
         f"{math.ceil(len(TRAIN) / BATCH)} pasos por época")

In [ ]:
# 3.2 Capa propia del modelo (necesaria para cargar los modelos guardados)
class GrisAleatorio(layers.Layer):
    """Convierte a escala de grises una fracción p de las imágenes (simula fotos infrarrojas nocturnas)."""
    def __init__(self, p=0.2, **kwargs):
        super().__init__(**kwargs)
        self.p = p
    def call(self, x, training=None):
        if not training:
            return x
        gris = x[..., 0:1] * 0.299 + x[..., 1:2] * 0.587 + x[..., 2:3] * 0.114
        gris = tf.concat([gris, gris, gris], axis=-1)
        m = tf.cast(tf.random.uniform([tf.shape(x)[0], 1, 1, 1]) < self.p, x.dtype)
        return m * gris + (1 - m) * x
    def get_config(self):
        return {**super().get_config(), "p": self.p}

---
# 4. Modelos del proyecto y sus probabilidades
Se cargan los dos EfficientNetV2-S entrenados y se usa **exactamente la misma división** con la que se entrenaron (`division_dataset.csv`).

In [ ]:
# 4.1 Cargar los modelos entrenados y verificar que la división de prueba sea idéntica
archivos = sorted(Path("/kaggle/input").rglob("profesor_*.keras"))   # solo los modelos grandes (no el estudiante)
por_carpeta = {}
for f in archivos:
    if "respaldo" in str(f):
        continue
    por_carpeta.setdefault(f.parent, []).append(f)
RUTAS_MODELOS = []
for carpeta, fs in por_carpeta.items():
    preferido = [f for f in fs if f.name == "profesor_mejor.keras"] or [f for f in fs if f.name == "profesor_final.keras"] or fs
    RUTAS_MODELOS.append(preferido[0])
assert len(RUTAS_MODELOS) >= 2, f"Se necesitan al menos 2 modelos y encontré {len(RUTAS_MODELOS)}: {archivos}"

def nombre_modelo(ruta):
    partes = [p for p in ruta.parts if p not in ("kaggle", "input", "/", "entrenamiento", ruta.name)]
    return "/".join(partes[-2:]) if partes else ruta.stem

MODELOS = {}
for r in RUTAS_MODELOS:
    MODELOS[nombre_modelo(r)] = keras.models.load_model(r, custom_objects={"GrisAleatorio": GrisAleatorio})
    log.info(f"Modelo cargado: {nombre_modelo(r)}  ({r}, {r.stat().st_size / 1e6:.0f} MB)")

# La división se toma EXACTAMENTE de la que usaron los modelos al entrenar (division_dataset.csv de cada modelo).
# Así no depende de versiones de librerías de Kaggle: es la misma división, archivo por archivo.
divs = {f: pd.read_csv(f) for f in sorted(Path("/kaggle/input").rglob("division_dataset.csv"))}
assert divs, "No encontré division_dataset.csv en las entradas"
ref_f, ref = next(iter(divs.items()))
ref = ref.set_index("archivo")["split"]
for f, d in divs.items():
    s = d.set_index("archivo")["split"]
    igual = all(set(s[s == g].index) == set(ref[ref == g].index) for g in ["val", "test"])
    log.info(f"División de {f.parts[-3]}: {'✓ igual' if igual else '✗ DIFERENTE'} a la de {ref_f.parts[-3]}")
    assert igual, "Los modelos de entrada no se entrenaron con la misma división"
meta["split"] = meta["archivo"].map(ref).fillna("sin_usar")
n_tr = meta[meta.split == "train"].groupby("y").size().reindex(range(N_CLASES), fill_value=1)
PESOS = np.clip(n_tr.sum() / (N_CLASES * n_tr), 0.5, 5.0)
meta["peso"] = meta["y"].map(PESOS)
TRAIN = meta[meta.split == "train"].reset_index(drop=True)
VAL = meta[meta.split == "val"].reset_index(drop=True)
TEST = meta[meta.split == "test"].reset_index(drop=True)
ds_train, ds_val, ds_test = hacer_ds(TRAIN, entrenamiento=True), hacer_ds(VAL), hacer_ds(TEST)
log.info(f"Se usa la división EXACTA del entrenamiento: train {len(TRAIN)} | val {len(VAL)} | test {len(TEST)}")

In [ ]:
# 4.2 Probabilidades de cada modelo en validación y prueba (foto original y volteada)
def probas(modelo, ds, volteo=False):
    size = modelo.input_shape[1]                      # cada modelo recibe las fotos a su propio tamaño
    if size != IMG_SIZE:
        ds = ds.map(lambda x, y: (tf.image.resize(x, (size, size), antialias=True), y))
    if volteo:
        ds = ds.map(lambda x, y: (tf.image.flip_left_right(x), y))
    return tf.nn.softmax(modelo.predict(ds, verbose=0)).numpy()

P = {}
for nombre, m in MODELOS.items():
    t0 = time.time()
    for grupo, ds in [("val", ds_val), ("test", ds_test)]:
        P[(nombre, grupo, "orig")] = probas(m, ds)
        P[(nombre, grupo, "flip")] = probas(m, ds, volteo=True)
    log.info(f"Predicciones de {nombre} listas ({time.time() - t0:.0f} s)")
Y = {"val": VAL["y"].values, "test": TEST["y"].values}
acc = lambda p, g: float((p.argmax(1) == Y[g]).mean())

---
# 5. Ensamble por promedio, ensamble ponderado y modelo híbrido
- **Promedio:** p = 0,5·p₁ + 0,5·p₂
- **Ponderado:** p = w·p₁ + (1 − w)·p₂, con w de 0,1 a 0,9 y con o sin TTA (foto original + volteada). Gana el mejor en **validación**.
- **Híbrido:** p(especie | foto, lugar) ∝ p(especie | foto) × [p(especie | departamento) / p(especie)]^λ. Las frecuencias por departamento salen **solo del entrenamiento**; α y λ se eligen con validación.

In [ ]:
# 5.1 Ensamble por promedio y ensamble ponderado (pesos y TTA elegidos con validación)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
nombres = list(MODELOS)
assert len(nombres) == 2, f"Se esperaban 2 modelos grandes y hay {len(nombres)}: {nombres}"

def combinar(pesos, grupo, tta):
    p = 0
    for w, n in zip(pesos, nombres):
        pn = P[(n, grupo, "orig")]
        if tta:
            pn = (pn + P[(n, grupo, "flip")]) / 2
        p = p + w * pn
    return p / sum(pesos)

def metricas(p, g):
    yt, yp = Y[g], p.argmax(1)
    _, _, f1, _ = precision_recall_fscore_support(yt, yp, labels=range(N_CLASES), average="macro", zero_division=0)
    return {"acc": accuracy_score(yt, yp), "f1": f1, "top3": float((np.argsort(-p, 1)[:, :3] == yt[:, None]).any(1).mean())}

grid_w = pd.DataFrame([{"w_v2": a, "w_v4": round(1 - a, 1), "TTA": t, "val_accuracy": acc(combinar((a, 1 - a), "val", t), "val")}
                       for t in (False, True) for a in np.round(np.arange(0.1, 1.0, 0.1), 1)])
grid_w = grid_w.sort_values("val_accuracy", ascending=False, kind="mergesort").reset_index(drop=True)
display(grid_w.head(8).round(4))
W_POND, TTA_POND = (float(grid_w.iloc[0]["w_v2"]), float(grid_w.iloc[0]["w_v4"])), bool(grid_w.iloc[0]["TTA"])
log.info(f"Ponderado ELEGIDO CON VALIDACIÓN: pesos {W_POND} | TTA {'sí' if TTA_POND else 'no'} | val {grid_w.iloc[0]['val_accuracy']:.4f}")

PROB = {}
for g in ("val", "test"):
    PROB[("v2", g)] = combinar((1, 0), g, False)
    PROB[("v4", g)] = combinar((0, 1), g, False)
    PROB[("promedio", g)] = combinar((0.5, 0.5), g, False)
    PROB[("ponderado", g)] = combinar(W_POND, g, TTA_POND)

In [ ]:
# 5.2 Modelo híbrido: ensamble ponderado + ubicación de la foto
import unicodedata
def norm_dep(s):
    def f(x):
        if not isinstance(x, str) or not x.strip():
            return "desconocido"
        x = "".join(c for c in unicodedata.normalize("NFD", x.lower()) if unicodedata.category(c) != "Mn")
        for q in ["departamento de ", "departamento del ", "depto. de ", "depto ", "departamento "]:
            x = x.replace(q, "")
        return x.strip()
    return s.map(f)
DEP = {"train": norm_dep(TRAIN["departamento"]), "val": norm_dep(VAL["departamento"]), "test": norm_dep(TEST["departamento"])}
CONTEO = pd.crosstab(DEP["train"], TRAIN["y"]).reindex(columns=range(N_CLASES), fill_value=0).drop(index="desconocido", errors="ignore")

def con_lugar(p, deps, alpha, lam):
    p_dep = (CONTEO + alpha).div(CONTEO.sum(1) + alpha * N_CLASES, axis=0)
    glob = CONTEO.sum(0) + alpha
    r = p_dep / (glob / glob.sum())
    f = np.vstack([r.loc[d].values if d in r.index else np.ones(N_CLASES) for d in deps])
    q = p * f ** lam
    return q / q.sum(1, keepdims=True)

def elegir_lugar(p_val):
    g = sorted(((acc(con_lugar(p_val, DEP["val"], a, l), "val"), -l, a, l) for a in [1, 3, 10] for l in [0, 0.25, 0.5, 0.75, 1.0]),
               reverse=True)
    return g[0][2], g[0][3], g[0][0]

A_H, L_H, v = elegir_lugar(PROB[("ponderado", "val")])
for g in ("val", "test"):
    PROB[("hibrido", g)] = con_lugar(PROB[("ponderado", g)], DEP[g], A_H, L_H)
log.info(f"Híbrido ELEGIDO CON VALIDACIÓN: alpha={A_H} lambda={L_H} | val {v:.4f}")

---
# 6. Modelo destilado (el de la app) y medición de tamaño y latencia
El modelo destilado se evalúa **en su versión TFLite float16**, el mismo archivo `mamiferos.tflite` que lleva la app, con las mismas fotos de validación y prueba.

In [ ]:
# 6.1 Modelo destilado en TFLite (el mismo archivo de la app): probabilidades y latencia
cand = sorted(Path("/kaggle/input").rglob("mamiferos.tflite")) or sorted(Path("/kaggle/input").rglob("*float16*.tflite"))
assert cand, "No encuentro mamiferos.tflite: agrega como Input la salida de notebook901900824d o un dataset con ese archivo"
RUTA_TFL = cand[0]
IMG_EST = 224
def correr_tflite(ruta, df, n_lat=150):
    it = tf.lite.Interpreter(model_path=str(ruta), num_threads=4); it.allocate_tensors()
    e, s = it.get_input_details()[0], it.get_output_details()[0]
    salida, tiempos = [], []
    for i, r in enumerate(df["ruta"]):
        x = leer_imagen(r, IMG_EST).numpy()[None].astype(e["dtype"])
        t0 = time.perf_counter(); it.set_tensor(e["index"], x); it.invoke(); q = it.get_tensor(s["index"])[0].copy()
        if i < n_lat: tiempos.append((time.perf_counter() - t0) * 1000)
        salida.append(q)
    p = np.array(salida, dtype="float32")
    if p.min() < 0 or abs(p.sum(1).mean() - 1) > 1e-2:            # por si la salida fueran logits
        p = tf.nn.softmax(p).numpy()
    return p, float(np.median(tiempos[5:]))
t0 = time.time()
PROB[("destilado", "val")], _ = correr_tflite(RUTA_TFL, VAL)
PROB[("destilado", "test")], LAT_TFL = correr_tflite(RUTA_TFL, TEST)
A_D, L_D, v = elegir_lugar(PROB[("destilado", "val")])
for g in ("val", "test"):
    PROB[("destilado_lugar", g)] = con_lugar(PROB[("destilado", g)], DEP[g], A_D, L_D)
log.info(f"Destilado TFLite ({RUTA_TFL.stat().st_size / 1e6:.1f} MB): val {acc(PROB[('destilado', 'val')], 'val'):.4f} | "
         f"con ubicación alpha={A_D} lambda={L_D} val {v:.4f} | latencia {LAT_TFL:.1f} ms | {time.time() - t0:.0f} s")

In [ ]:
# 6.2 Latencia en CPU de los modelos grandes (una foto a la vez) y del ajuste por ubicación
def latencia_keras(m, n=40):
    ts = []
    with tf.device("/CPU:0"):
        for r in TEST["ruta"].iloc[:n]:
            x = leer_imagen(r, m.input_shape[1])[None]
            t0 = time.perf_counter(); m(x, training=False); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts[5:]))
LAT = {n: latencia_keras(m) for n, m in MODELOS.items()}
TAM = {n: r.stat().st_size / 1e6 for n, r in zip(MODELOS, RUTAS_MODELOS)}
t0 = time.perf_counter(); con_lugar(PROB[("ponderado", "test")][:200], DEP["test"][:200], A_H, L_H)
LAT_LUGAR = (time.perf_counter() - t0) * 1000 / 200
TAM_LUGAR = len(json.dumps(CONTEO.values.tolist())) / 1e6
n2, n4 = nombres
lat_ens = (LAT[n2] + LAT[n4])
log.info(f"Latencia CPU: {n2} {LAT[n2]:.0f} ms | {n4} {LAT[n4]:.0f} ms | ubicación {LAT_LUGAR:.3f} ms | TFLite {LAT_TFL:.1f} ms")

---
# 7. Tabla comparativa y conclusión

In [ ]:
# 7.0 Tabla comparativa de escenarios
TAM_TFL = RUTA_TFL.stat().st_size / 1e6
x2 = 2 if TTA_POND else 1                    # con TTA cada modelo se ejecuta dos veces (foto original y volteada)
ESCENARIOS = [
    ("v2", f"Modelo individual v2 (EfficientNetV2-S)", TAM[n2], LAT[n2], False),
    ("v4", f"Modelo individual v4 (EfficientNetV2-S)", TAM[n4], LAT[n4], False),
    ("promedio", "Ensamble por promedio (0,5 / 0,5)", TAM[n2] + TAM[n4], lat_ens, False),
    ("ponderado", f"Ensamble ponderado ({W_POND[0]:.1f} / {W_POND[1]:.1f}{' + TTA' if TTA_POND else ''})".replace(".", ","), TAM[n2] + TAM[n4], lat_ens * x2, False),
    ("hibrido", "Modelo híbrido: ensamble ponderado + ubicación", TAM[n2] + TAM[n4] + TAM_LUGAR, lat_ens * x2 + LAT_LUGAR, False),
    ("destilado", "Modelo destilado MobileNetV3 (TFLite float16)", TAM_TFL, LAT_TFL, True),
    ("destilado_lugar", "Destilado + ubicación (la app)", TAM_TFL + TAM_LUGAR, LAT_TFL + LAT_LUGAR, True),
]
filas = []
for clave, nombre, tam, lat, cabe in ESCENARIOS:
    mv, mt = metricas(PROB[(clave, "val")], "val"), metricas(PROB[(clave, "test")], "test")
    filas.append({"clave": clave, "escenario": nombre, "val_acc": mv["acc"], "test_acc": mt["acc"], "test_f1": mt["f1"],
                  "test_top3": mt["top3"], "tam_mb": tam, "lat_ms": lat, "cabe": cabe, "en_app": clave == "destilado_lugar"})
TABLA = pd.DataFrame(filas)
TABLA.to_csv(EVI / "tabla_ensamble_vs_destilado.csv", index=False, encoding="utf-8-sig")
display(TABLA.drop(columns=["clave", "en_app"]).set_index("escenario").style.format(
    {"val_acc": "{:.2%}", "test_acc": "{:.2%}", "test_f1": "{:.3f}", "test_top3": "{:.2%}", "tam_mb": "{:.1f}", "lat_ms": "{:.1f}"})
    .highlight_max(subset=["test_acc", "test_f1"], color="#d4edda").highlight_min(subset=["tam_mb", "lat_ms"], color="#d4edda"))

fig, axs = plt.subplots(1, 3, figsize=(19, 4.8))
col = ["#9bbbd4", "#9bbbd4", "#2b7bba", "#2b7bba", "#1b4f8a", "#7ac58a", "#2e9e44"]
for ax, (c, tit, k) in zip(axs, [("test_acc", "Accuracy en prueba (%)", 100), ("tam_mb", "Tamaño (MB, escala log)", 1),
                                   ("lat_ms", "Latencia CPU por foto (ms, escala log)", 1)]):
    v = TABLA[c] * k; ax.barh(TABLA["escenario"], v, color=col); ax.invert_yaxis(); ax.set_title(tit)
    for i, val in enumerate(v): ax.text(val, i, f" {val:.1f}", va="center", fontsize=9)
axs[0].set_xlim(85, 100); axs[0].axvline(META_ACCURACY * 100, ls="--", c="red", label="meta 94 %"); axs[0].legend()
for ax in axs[1:]: ax.set_xscale("log"); ax.set_yticklabels([])
plt.tight_layout(); plt.savefig(EVI / "ensamble_vs_destilado.png", dpi=120); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for (_, r), c in zip(TABLA.iterrows(), col):
    ax.scatter(r["lat_ms"], r["test_acc"] * 100, s=40 + r["tam_mb"] / 2, color=c, edgecolor="black", zorder=3)
    ax.annotate(r["escenario"].split(" (")[0], (r["lat_ms"], r["test_acc"] * 100), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("latencia por foto en CPU (ms, escala log)"); ax.set_ylabel("accuracy en prueba (%)")
ax.set_title("Acierto vs. velocidad (tamaño del punto = tamaño del modelo)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(EVI / "acierto_vs_latencia.png", dpi=120); plt.show()

In [ ]:
# 7.1 Funciones del PDF: tabla comparativa + conclusión de media página
import textwrap
from matplotlib.backends.backend_pdf import PdfPages
def _pct(x): return "—" if pd.isna(x) else f"{x * 100:.2f} %".replace(".", ",")
def _num(x, d=1): return "—" if pd.isna(x) else f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def _f1(x): return "—" if pd.isna(x) else f"{x:.3f}".replace(".", ",")

def crear_pdf(tabla, conclusion, ruta, nota=""):
    cols = [("Escenario", 0.30), ("Accuracy\nvalidación", 0.10), ("Accuracy\nprueba", 0.10), ("F1 macro\nprueba", 0.09),
            ("Top-3\nprueba", 0.09), ("Tamaño\n(MB)", 0.09), ("Latencia\nCPU (ms/foto)", 0.12), ("¿Cabe en\nel celular?", 0.11)]
    with PdfPages(ruta) as pdf:
        fig = plt.figure(figsize=(8.27, 11.69))                                   # A4 vertical
        fig.text(0.06, 0.955, "Ensamble frente al modelo destilado", fontsize=17, weight="bold", color="#1b4332")
        fig.text(0.06, 0.935, "Mamíferos de Colombia · Jaime Andrés Ortega Cadena · David Alejandro Muñoz Carmona",
                 fontsize=8.8, color="#444444")
        fig.text(0.06, 0.905, "1. Tabla comparativa de escenarios (2.458 fotos de prueba de Colombia, nunca vistas)",
                 fontsize=10.5, weight="bold", color="#1b4332")
        x0, ancho, y = 0.06, 0.88, 0.89
        xs = [x0 + sum(w for _, w in cols[:i]) * ancho for i in range(len(cols))]
        fig.patches.append(plt.Rectangle((x0, y - 0.034), ancho, 0.034, transform=fig.transFigure, color="#1b4332"))
        for (t, _), x in zip(cols, xs):
            fig.text(x + 0.004, y - 0.017, t, fontsize=7.3, weight="bold", color="white", va="center", linespacing=1.15)
        y -= 0.034
        for i, (_, r) in enumerate(tabla.iterrows()):
            nombre = "\n".join(textwrap.wrap(r["escenario"], 34)); n = nombre.count("\n") + 1; alto = 0.0135 * n + 0.012
            app = r.get("en_app", False)
            color = "#d8eedd" if app else ("#f2f5f2" if i % 2 == 0 else "white")
            fig.patches.append(plt.Rectangle((x0, y - alto), ancho, alto, transform=fig.transFigure, color=color))
            vals = [nombre, _pct(r["val_acc"]), _pct(r["test_acc"]), _f1(r["test_f1"]), _pct(r["test_top3"]),
                    _num(r["tam_mb"]), _num(r["lat_ms"]), "Sí" if r["cabe"] else "No"]
            for j, (v, x) in enumerate(zip(vals, xs)):
                fig.text(x + 0.004, y - alto / 2, v, fontsize=7.6, va="center", linespacing=1.2,
                         weight="bold" if (app or j == 0) else "normal", color="#1b4332" if app else "#222222")
            y -= alto
        pie = "\n".join(textwrap.wrap(
            "En verde, el escenario que queda en la app. Latencia medida en CPU, una foto a la vez (como en un celular). "
            "Ensamble = 2 EfficientNetV2-S (260 px). Destilado = MobileNetV3-Large (224 px) en TFLite float16. "
            "Ubicación = ajuste por departamento con α y λ elegidos con validación. " + nota, 135))
        fig.text(0.06, y - 0.012, pie, fontsize=6.9, color="#555555", va="top", linespacing=1.3)
        y -= 0.012 + 0.0115 * (pie.count("\n") + 1) + 0.03
        fig.text(0.06, y, "2. Conclusión: qué modelo se queda en la app y por qué", fontsize=10.5, weight="bold", color="#1b4332")
        y -= 0.016
        for parrafo in conclusion:
            t = "\n".join(textwrap.wrap(parrafo, 112))
            fig.text(0.06, y, t, fontsize=8.6, va="top", linespacing=1.42, color="#222222")
            y -= 0.0158 * (t.count("\n") + 1) + 0.012
        pdf.savefig(fig); plt.close(fig)

In [ ]:
# 7.2 Conclusión de media página (se escribe con los números de esta ejecución)
def fila(clave): return TABLA.set_index("clave").loc[clave]
E, H, Dd, A = fila("ponderado"), fila("hibrido"), fila("destilado"), fila("destilado_lugar")
PR = fila("promedio")
CONCLUSION = [
    f"Combinar modelos sí mejora el acierto. El ensamble por promedio llega a {_pct(PR['test_acc'])} en prueba y el ponderado "
    f"(pesos elegidos con validación) a {_pct(E['test_acc'])}, frente a {_pct(fila('v4')['test_acc'])} del mejor modelo individual. "
    f"El modelo híbrido, que suma a la imagen la información de dónde se tomó la foto, es el más preciso: {_pct(H['test_acc'])} "
    f"de accuracy y {_f1(H['test_f1'])} de F1 macro.",
    f"Pero el ensamble pesa {_num(E['tam_mb'])} MB y tarda {_num(E['lat_ms'])} ms por foto en CPU. No cabe en un celular "
    f"de gama media y obligaría a enviar cada foto a un servidor, algo imposible en campo, donde casi nunca hay señal.",
    f"Por eso en la app se queda el modelo destilado MobileNetV3-Large en TFLite float16 con el ajuste por ubicación: pesa "
    f"{_num(A['tam_mb'])} MB ({E['tam_mb'] / A['tam_mb']:.0f} veces menos), clasifica una foto en {_num(A['lat_ms'])} ms "
    f"({E['lat_ms'] / A['lat_ms']:.0f} veces más rápido) y funciona sin internet. Su acierto es {_pct(A['test_acc'])}, "
    f"solo {(H['test_acc'] - A['test_acc']) * 100:.1f} puntos por debajo del híbrido con ensamble. La ubicación le suma "
    f"{(A['test_acc'] - Dd['test_acc']) * 100:.1f} puntos casi sin costo: es una tabla de 40 factores por departamento."
    "El ensamble no se descarta: es el profesor que le enseñó al modelo destilado y puede usarse en el panel web para revisar "
    "las fotos con confianza baja que lleguen de campo. Además, en la app MegaDetector revisa primero si hay un animal, y "
    "cuando la confianza es baja se muestran las tres especies más probables para que la persona confirme.",
]
for p in CONCLUSION: print(p, "\n")

In [ ]:
# 7.3 Genera el PDF (tabla + conclusión de media página)
ruta_pdf = SALIDA / "ensamble_vs_destilado.pdf"
crear_pdf(TABLA, CONCLUSION, ruta_pdf)
log.info(f"PDF listo: {ruta_pdf}")